# Combining Two Datasets - Kyla Duran

I encountered difficulties downloading the Degree Heating Week (DHW) data from NOAA's Coral Reef Watch dataset.

To work around this, I ran four different download methods simultaneously. One method succeeded for the majority of the timeline but excluded 1986, while another successfully captured only the 1986–1990 data. I combined these two outputs to create a complete dataset spanning 1986–2026.


In [3]:
import requests #allows Python to interact with the web
import zipfile #allows python to open and extract compressed zip folders
import pandas as pd

url = "https://github.com/kduran-stack/climate_collection_kd/raw/refs/heads/clean-data/data/hollywood_reefs_1987_2026_all_files.zip"

#wjere file will be saved in Google colab
zip_path = "/content/hollywood_reefs_1987_2026_all_files.zip"

response = requests.get(url) #sending a web request to download the file to colab
response.raise_for_status() # Check if the download succeeded (HTTP status 200). If the link is broken (e.g., 404 error), stop immediately so we don't save an error page as a fake ZIP file.

# Save the downloaded bytes from memory (RAM) onto Colab's hard drive so it exists as a real file.
# ("wb" = Write Binary, which creates a new empty file at zip_path and writes the raw ZIP data into it)
with open(zip_path, "wb") as f:
    f.write(response.content)
# Open the saved ZIP file in read mode and extract all 45 CSV files
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/hollywood_reefs")

print("Download and extraction complete!")

url = "https://raw.githubusercontent.com/kduran-stack/climate_collection_kd/refs/heads/clean-data/data/dhw_1986_1990.csv"

df = pd.read_csv(url)

print(df.head())

Download and extraction complete!
                   time       latitude     longitude        CRW_DHW  \
0                   UTC  degrees_north  degrees_east  Celsius weeks   
1  1986-01-01T12:00:00Z         26.125       -80.075            0.0   
2  1986-01-02T12:00:00Z         26.125       -80.075            0.0   
3  1986-01-03T12:00:00Z         26.125       -80.075            0.0   
4  1986-01-04T12:00:00Z         26.125       -80.075            0.0   

           CRW_HOTSPOT  CRW_BAA  
0              Celsius        1  
1                -5.45        0  
2                -5.28        0  
3                -5.07        0  
4  -5.2700000000000005        0  


In [4]:
import os
import zipfile
from google.colab import files

# Unpacking the zip files so pandas can read each csv

#zip path copied from google colab files folder
zip_path = "/content/hollywood_reefs_1987_2026_all_files.zip"
#new name of folder that the extracted files will be put into to
extract_dir = "/content/hollywood_reefs_extracted"

#unpacking the zipfile
with zipfile.ZipFile(zip_path, "r") as zip_ref:
  zip_ref.extractall(extract_dir)

# taking the master csv and setting it aside in the new folder
master_csv_path = os.path.join(
    extract_dir, "hollywood_reefs_1987_2026_dhw_master.csv"
)

# reading the csv with pandas so that it can loaded into RAM
df_1987_2026 = pd.read_csv(master_csv_path)



In [5]:

df_1987_2026.head()

,time,latitude,longitude,CRW_DHW,CRW_HOTSPOT,CRW_BAA
0,1987-01-01 00:00:00+00:00,26.125,-80.075,0.0,-4.28,0
1,1987-01-02 00:00:00+00:00,26.125,-80.075,0.0,-4.39,0
2,1987-01-03 00:00:00+00:00,26.125,-80.075,0.0,-4.19,0
3,1987-01-04 00:00:00+00:00,26.125,-80.075,0.0,-4.47,0
4,1987-01-05 00:00:00+00:00,26.125,-80.075,0.0,-4.87,0


In [6]:

# reading the csv with pandas so that it can loaded into RAM
df_1986_1990 = pd.read_csv("/content/dhw_1986_1990.csv")
df_1986_1990.head()


,time,latitude,longitude,CRW_DHW,CRW_HOTSPOT,CRW_BAA
0,UTC,degrees_north,degrees_east,Celsius weeks,Celsius,1
1,1986-01-01T12:00:00Z,26.125,-80.075,0.0,-5.45,0
2,1986-01-02T12:00:00Z,26.125,-80.075,0.0,-5.28,0
3,1986-01-03T12:00:00Z,26.125,-80.075,0.0,-5.07,0
4,1986-01-04T12:00:00Z,26.125,-80.075,0.0,-5.2700000000000005,0


In [10]:

# reading the csv with pandas so that it can loaded into RAM
df_1986_1990 = pd.read_csv("/content/dhw_1986_1990.csv", skiprows=[1])
df_1986_1990.head()


,time,latitude,longitude,CRW_DHW,CRW_HOTSPOT,CRW_BAA
0,1986-01-01T12:00:00Z,26.125,-80.075,0.0,-5.45,0
1,1986-01-02T12:00:00Z,26.125,-80.075,0.0,-5.28,0
2,1986-01-03T12:00:00Z,26.125,-80.075,0.0,-5.07,0
3,1986-01-04T12:00:00Z,26.125,-80.075,0.0,-5.27,0
4,1986-01-05T12:00:00Z,26.125,-80.075,0.0,-5.25,0


In [11]:

#After checking the header don't need this since data is already clean


# Filter to keep only 1986
df_1986_1990["time"] = pd.to_datetime(df_1986_1990["time"])
df_1986 = df_1986_1990[df_1986_1990["time"].dt.year == 1986].copy()

#Align formats and combine datasets
df_1987_2026["time"] = pd.to_datetime(df_1987_2026["time"])

common_cols = [col for col in df_1986.columns if col in df_1987_2026.columns]
df_combined = pd.concat( #combining the dfs
    [df_1986[common_cols], df_1987_2026[common_cols]], ignore_index=True
)

# Convert numeric columns to proper numbers and sort chronologically

#numeric_cols - convert the contents of these columns into numbers so I can do calculations or create graphs
#does not include "time", bc I want this column to be treated as a datetime object so that the data can be sorted chronologically
numeric_cols = ["latitude", "longitude", "CRW_DHW", "CRW_HOTSPOT", "CRW_BAA"]
for col in numeric_cols:
  if col in df_combined.columns:
    df_combined[col] = pd.to_numeric(df_combined[col], errors="coerce") #errors="coerce", when python encounters something that cannot be converted into a number such as "error or N/A" it will replace it with NaN instead of crashing

#putting the combined df in chronological order
df_combined = df_combined.sort_values("time").reset_index(drop=True)


final_output_path = "/content/dhw_1986_2026_kd_.csv" #where to save in google colab
df_combined.to_csv(final_output_path, index=False) #index false since it is already in chronological order

print(f"Saved: {final_output_path}")
print(f"Date range: {df_combined['time'].min()} to {df_combined['time'].max()}")
print(f"Total rows: {len(df_combined)}")

files.download(final_output_path)


Saved: /content/dhw_1986_2026_kd_.csv
Date range: 1986-01-01 12:00:00+00:00 to 2026-11-17 00:00:00+00:00
Total rows: 14919


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>